In [ ]:
import os
os.environ['HF_HOME'] = '/media/my_drives/DATA4/models'
os.environ['TRANSFORMERS_CACHE'] = '/media/my_drives/DATA4/models'
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.chdir('/media/my_drives')
import pandas as pd
from tqdm import tqdm
import random
import numpy as np
import re
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
random.seed(1)

/media/my_drives/DATA4/environments/max/ImageBenchmark/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/my_drives/DATA4/environments/max/ImageBenchmark/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-09-30 14:37:40.914553: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759239460.926068 1234046 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759239460.929557 1234046 cuda_blas.cc:1407] Unable to regis

In [2]:
def resize_img(image, max_dim:int):
    width, height = image.size
    if width > height:
        new_width = max_dim
        new_height = int((new_width / width) * height)
    else:
        new_height = max_dim
        new_width = int((new_height / height) * width)
    return image.resize((new_width, new_height))

def is_openable(path):
    try:
        with Image.open(path):
            return True
    except IOError:
        return False

def add_drive_path(path):
    if path.startswith('/media/my_drives/DATA4/data/image_benchmark_phi') == False:
        return f'/media/my_drives/DATA4/data/image_benchmark_phi/{path}'
    else:
        return path

def is_actual_prediction(prediction: str):
    if any(word in prediction.lower() for word in ['sorry', 'cant']):
        return 'No prediction available'
    return prediction

## Phi

In [ ]:
model_path = "microsoft/Phi-4-multimodal-instruct"
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="cuda", 
    torch_dtype="auto", 
    trust_remote_code=True, 
    attn_implementation='flash_attention_2',
).cuda()

generation_config = GenerationConfig.from_pretrained(model_path)

#### Zero-Shot

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_phi4_zeroshot'

In [ ]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv')
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        prompt = f'<|user|><|image_1|>Assign which aesthetic score measured as a 5-point likert scale is best fitting for the image. The class ranges are: {classes}<|end|><|assistant|>'
    elif DATASET == 'GenImgNet':
        prompt = f'<|user|><|image_1|>Assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images. The class names are: {classes}<|end|><|assistant|>'
    elif DATASET == 'AIDA': 
        prompt = f'<|user|><|image_1|>Assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images. The class names are: {classes}<|end|><|assistant|>'
    else:
        prompt = f'<|user|><|image_1|>Assign the best fitting class to describe the image by choosing one of the following classes: {classes}<|end|><|assistant|>'

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        try:
            image1 = resize_img(Image.open(df.at[i, 'image_path']), 300)
        
            inputs = processor(text=prompt, images=[image1], return_tensors='pt').to('cuda')
            generate_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                generation_config=generation_config,
            )
            generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
            response = processor.batch_decode(
                generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
            df.at[i, PRED_COL] = response
        except Exception as e:
            print(str(e)[:300])

            
        if i % 50 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower()
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

#### Few-Shot

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_phi4_fewshot'

In [ ]:
def get_prompt_with_examples(base_prompt:str, df_example: pd.DataFrame):
    prompt = ''
    for i in df_example.index:
        prompt = f"{prompt}<|user|><|image_{i+1}|>{base_prompt}<|end|><|assistant|>'{df_example.at[i, 'label']}'<|end|>"
    prompt = f"{prompt}<|user|><|image_{df_example.shape[0]+1}|>{base_prompt}<|end|><|assistant|>"
    return prompt

In [ ]:
for DATASET in tqdm(datasets):
    df_path = f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv'
    df = pd.read_csv(df_path)
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        base_prompt = f'Assign which aesthetic score measured as a 5-point likert scale is best fitting for the image. The class ranges are: {classes}'
    elif DATASET == 'GenImgNet':
        base_prompt = f'Assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images. The class names are: {classes}'
    elif DATASET == 'AIDA': 
        base_prompt = f'Assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images. The class names are: {classes}'
    else:
        base_prompt = f'Assign the best fitting class to describe the image by choosing one of the following classes: {classes}'

    df_example_path = f'DATA4/max/Image_Benchmark/CSVs/{DATASET}.csv'
    assert df_example_path != df_path
    df_examples = pd.read_csv(df_example_path)
    df_examples['image_path'] = df_examples['image_path'].apply(add_drive_path)
    df_examples = df_examples.loc[~df_examples.image_path.isin(df.image_path.values)]
    df_examples = df.groupby('label').apply(lambda x: x.sample(1, random_state=1)).reset_index(drop=True)
    n_examples = len(df_examples.label.unique())

    prompt = get_prompt_with_examples(base_prompt, df_examples)

    example_images = []
    for index in df_examples.index:
        example_images.append(resize_img(Image.open(df_examples.at[index, 'image_path']), 300))

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        try:
            image_predict = resize_img(Image.open(df.at[i, 'image_path']), 300)
            images = example_images.copy()
            images.append(image_predict)
        
            inputs = processor(text=prompt, images=images, return_tensors='pt').to('cuda')
            generate_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                generation_config=generation_config,
            )
            generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
            response = processor.batch_decode(
                generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
            df.at[i, PRED_COL] = response
        except Exception as e:
            print(str(e)[:300])

            
        if i % 50 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower()
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

#### Combined Decision

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_phi4_combined'

In [ ]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv')
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        base_prompt = f'<|user|><|image_1|>You are given an image classification task. Your goal is to assign which aesthetic score measured as a 5-point likert scale is best fitting for the image.'
    elif DATASET == 'GenImgNet':
        base_prompt = f'<|user|><|image_1|>You are given an image classification task. Your goal is to assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images.'
    elif DATASET == 'AIDA': 
        base_prompt = f'<|user|><|image_1|>You are given an image classification task. Your goal is to assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images.'
    else:
        base_prompt = f'<|user|><|image_1|>You are given an image classification task. Your goal is to assign the most accurate class to the provided image.'

    #if PRED_COL not in df.columns:
    df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        try:
            image1 = resize_img(Image.open(df.at[i, 'image_path']), 300)

            prompt = f'''{base_prompt} You must choose exactly one label from the following list of possible classes: {classes}.

            Additional information is available:
            - Prior prediction of model without example guidance: {is_actual_prediction(df.at[i, 'prediction_phi4_zeroshot'])}  
            - Prior prediction of model with example guidance: {is_actual_prediction(df.at[i, 'prediction_phi4_fewshot'])}  
            
            Instructions: 
            1. First, carefully consider the content of the image itself.  
            2. Then, review the two model predictions as supporting evidence.  
            3. Based on both the image and the prior predictions, select the single class that best fits.  
            4. Respond with only the chosen class label, nothing else.<|end|><|assistant|>'''
        
            inputs = processor(text=prompt, images=[image1], return_tensors='pt').to('cuda')
            generate_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                generation_config=generation_config,
            )
            generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
            response = processor.batch_decode(
                generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
            df.at[i, PRED_COL] = response
        except Exception as e:
            print(str(e)[:300])

            
        if i % 50 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower()
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

## GPT-5

In [ ]:
from openai import OpenAI
import base64

os.environ["OPENAI_API_KEY"] = "API_KEY_HERE"
client = OpenAI()

#### Zero-Shot

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def predict_gpt(img_path, prompt):
    base64_image = encode_image(img_path)

    response = client.responses.create(
        model="gpt-5-2025-08-07",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
    )
    
    return response.output_text

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_gpt5_zeroshot'

In [ ]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv')
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        prompt = f'Assign which aesthetic score measured as a 5-point likert scale is best fitting for the image. The class ranges are: {classes}'
    elif DATASET == 'GenImgNet':
        prompt = f'Assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images. The class names are: {classes}'
    elif DATASET == 'AIDA': 
        prompt = f'Assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images. The class names are: {classes}'
    else:
        prompt = f'Assign the best fitting class to describe the image by choosing one of the following classes: {classes}'

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        df.at[i, PRED_COL] = predict_gpt(df.at[i, 'image_path'], prompt)

        if i % 10 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower().str.replace('"', "'")
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

#### Few-Shot

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def prepare_examples(prompt:str, df_examples:pd.DataFrame):
    messages = []
    for i in df_examples.index:  
        base64_image = encode_image(df_examples.at[i, 'image_path'])
        messages.append({
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        })
        messages.append({
                "role": "assistant",
                "content": [
                    {
                        "type": "output_text",
                        "text": f"'{df_examples.at[i, 'label']}'",
                    }
                ],
            })
    return messages

def predict_gpt(prompt:str, inference_img_path:str, messages_examples:list):
    messages = messages_examples.copy()
    base64_image = encode_image(inference_img_path)

    messages.append({
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        })


    response = client.responses.create(
        model="gpt-5-2025-08-07",
        input=messages,
    )
    
    return response.output_text

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_gpt5_fewshot'

In [ ]:
for DATASET in tqdm(datasets):
    df_path = f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv'
    df = pd.read_csv(df_path)
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        base_prompt = f'Assign which aesthetic score measured as a 5-point likert scale is best fitting for the image. The class ranges are: {classes}'
    elif DATASET == 'GenImgNet':
        base_prompt = f'Assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images. The class names are: {classes}'
    elif DATASET == 'AIDA': 
        base_prompt = f'Assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images. The class names are: {classes}'
    else:
        base_prompt = f'Assign the best fitting class to describe the image by choosing one of the following classes: {classes}'

    df_example_path = f'DATA4/max/Image_Benchmark/CSVs/{DATASET}.csv'
    assert df_example_path != df_path
    df_examples = pd.read_csv(df_example_path)
    df_examples['image_path'] = df_examples['image_path'].apply(add_drive_path)
    df_examples = df_examples.loc[~df_examples.image_path.isin(df.image_path.values)]
    df_examples = df.groupby('label').apply(lambda x: x.sample(1, random_state=1)).reset_index(drop=True)
    n_examples = len(df_examples.label.unique())

    example_messages = prepare_examples(base_prompt, df_examples)

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        df.at[i, PRED_COL] = predict_gpt(base_prompt, df.at[i, 'image_path'], example_messages)

        if i % 10 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower().str.replace('"', "'")
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

#### Combined Decision

In [6]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def predict_gpt(img_path, prompt):
    base64_image = encode_image(img_path)

    response = client.responses.create(
        model="gpt-5-2025-08-07",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
    )
    
    return response.output_text

In [7]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_gpt5_combined'

In [ ]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv')
    classes = list(df.label.unique())

    if DATASET == 'KonIQ':
        base_prompt = f'You are given an image classification task. Your goal is to assign which aesthetic score measured as a 5-point likert scale is best fitting for the image.'
    elif DATASET == 'GenImgNet':
        base_prompt = f'You are given an image classification task. Your goal is to assign whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images.'
    elif DATASET == 'AIDA': 
        base_prompt = f'You are given an image classification task. Your goal is to assign whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images.'
    else:
        base_prompt = f'You are given an image classification task. Your goal is to assign the most accurate class to the provided image.'

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        prompt = f'''{base_prompt} You must choose exactly one label from the following list of possible classes: {classes}.

            Additional information is available:
            - Prior prediction of model without example guidance: {is_actual_prediction(df.at[i, 'prediction_gpt5_zeroshot'])}  
            - Prior prediction of model with example guidance: {is_actual_prediction(df.at[i, 'prediction_gpt5_fewshot'])}  
            
            Instructions: 
            1. First, carefully consider the content of the image itself.  
            2. Then, review the two model predictions as supporting evidence.  
            3. Based on both the image and the prior predictions, select the single class that best fits.  
            4. Respond with only the chosen class label, nothing else.'''
        
        df.at[i, PRED_COL] = predict_gpt(df.at[i, 'image_path'], prompt)

        if i % 10 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

    # clean up class names
    conditions = []
    values = []
    for cl in classes:
        conditions.append(df[PRED_COL].str.contains(rf"'{re.escape(cl.lower())}'", case=False, na=False, regex=True)) # match only with surrounding single quotation marks, so that female is not matched to male etc.
        values.append(cl)
    df[PRED_COL] = df[PRED_COL].str.lower().str.replace('"', "'")
    df[PRED_COL] = np.select(conditions, values, default=df[PRED_COL].str.lower().str.replace("'", ""))
    
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv', index=False)

  6%|██▏                                     | 1/18 [22:15<6:18:20, 1335.31s/it]/tmp/ipykernel_1234046/1232005075.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Chanel' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, PRED_COL] = predict_gpt(df.at[i, 'image_path'], prompt)
